In [2]:

import pandas as pd
import time

# Función para formatear el tiempo en mm:ss.ms
def formatear_tiempo(segundos):
    minutos = int(segundos // 60)
    segundos_restantes = segundos % 60
    return f"{minutos:02d}:{segundos_restantes:05.2f}"

# Variables iniciales
tiempo_inicio = None
tiempo_vuelta_inicio = None
tiempo_vuelta_acumulado = 0
en_ejecucion = False

# Tiempo total global
tiempo_total_acumulado = 0

# Diccionario para almacenar pilotos y sus vueltas
# Estructura: { "Piloto": [ (vuelta, tiempo_vuelta, diferencia, tiempo_global) ] }
pilotos = {}

# Selección inicial de piloto
piloto_actual = input("Ingrese el nombre del piloto inicial: ")
pilotos[piloto_actual] = []

# Menú
print("\n== Cronómetro para registrar vueltas ==")
print("Comandos:")
print("  [1] Iniciar/Reanudar")
print("  [2] Registrar vuelta")
print("  [3] Detener (pausar)")
print("  [4] Reiniciar (solo vueltas del piloto actual)")
print("  [5] Salir y guardar archivo Excel")
print("  [6] Cambiar de piloto\n")

# Bucle principal
while True:
    comando = input(">> ")

    if comando == '1':  # Iniciar/Reanudar
        if not en_ejecucion:
            if tiempo_inicio is None:
                tiempo_inicio = time.time()
            tiempo_vuelta_inicio = time.time()
            en_ejecucion = True
            print(f"Cronómetro en marcha para {piloto_actual}.")
        else:
            print("El cronómetro ya está en marcha.")

    elif comando == '2':  # Registrar vuelta
        if en_ejecucion:
            tiempo_actual = time.time()
            tiempo_vuelta_acumulado += tiempo_actual - tiempo_vuelta_inicio
            tiempo_total_acumulado += tiempo_vuelta_acumulado

            # Convertir vuelta en segundos para cálculos
            vuelta_segundos = tiempo_vuelta_acumulado

            # Calcular diferencia con vuelta anterior
            if pilotos[piloto_actual]:
                ultima_vuelta = pilotos[piloto_actual][-1][1]  # tiempo anterior en string
                # Convertimos el string de la última vuelta a segundos
                ult_min, ult_sec = map(float, ultima_vuelta.split(":"))
                ultimos_segundos = ult_min * 60 + ult_sec
                diferencia = vuelta_segundos - ultimos_segundos
            else:
                diferencia = 0  # primera vuelta, no hay diferencia

            vuelta_num = len(pilotos[piloto_actual]) + 1
            pilotos[piloto_actual].append((
                vuelta_num,
                formatear_tiempo(vuelta_segundos),
                diferencia,
                formatear_tiempo(tiempo_total_acumulado)
            ))

            signo = "+" if diferencia > 0 else "-"
            print(f"{piloto_actual} - Vuelta {vuelta_num}: {formatear_tiempo(vuelta_segundos)} "
                  f"({signo}{abs(diferencia):.2f}s) | Tiempo global: {formatear_tiempo(tiempo_total_acumulado)}")

            tiempo_vuelta_acumulado = 0
            tiempo_vuelta_inicio = time.time()
        else:
            print("Debes iniciar el cronómetro primero con '1'.")

    elif comando == '3':  # Detener (pausar)
        if en_ejecucion:
            tiempo_actual = time.time()
            tiempo_vuelta_acumulado += tiempo_actual - tiempo_vuelta_inicio
            en_ejecucion = False
            print("Cronómetro detenido (pausa).")
        else:
            print("El cronómetro no está en marcha.")

    elif comando == '4':  # Reiniciar (solo vueltas del piloto actual)
        pilotos[piloto_actual] = []
        tiempo_vuelta_acumulado = 0
        en_ejecucion = False
        print(f"Vueltas reiniciadas para {piloto_actual}. El tiempo global sigue intacto.")

    elif comando == '5':  # Salir y exportar a Excel
        break

    elif comando == '6':  # Cambiar de piloto
        piloto_actual = input("Ingrese el nombre del nuevo piloto: ")
        if piloto_actual not in pilotos:
            pilotos[piloto_actual] = []
        tiempo_vuelta_acumulado = 0
        en_ejecucion = False
        print(f"Ahora registrando tiempos para {piloto_actual}. El tiempo global sigue acumulando.")

    else:
        print("Comando no reconocido.")

# Exportar a Excel y determinar mejor piloto/vuelta
if any(pilotos.values()):
    data = []
    mejor_piloto = None
    mejor_vuelta = None
    mejor_tiempo = float("inf")

    for piloto, vueltas in pilotos.items():
        for v in vueltas:
            data.append([piloto, v[0], v[1], f"{v[2]:.2f}s", v[3]])

            # Buscar la vuelta más rápida
            min_, sec_ = map(float, v[1].split(":"))
            segundos_vuelta = min_ * 60 + sec_
            if segundos_vuelta < mejor_tiempo:
                mejor_tiempo = segundos_vuelta
                mejor_piloto = piloto
                mejor_vuelta = v[0]

    df = pd.DataFrame(data, columns=["Piloto", "Número de vuelta", "Tiempo por vuelta", "Diferencia", "Tiempo global"])
    nombre_archivo = "vueltas_pilotos.xlsx"
    df.to_excel(nombre_archivo, index=False)

    print(f"\nDatos exportados exitosamente a '{nombre_archivo}'")
    print(f"El mejor piloto fue {mejor_piloto} en la vuelta {mejor_vuelta} con un tiempo de {formatear_tiempo(mejor_tiempo)}")

else:
    print("No se registraron vueltas.")


ModuleNotFoundError: No module named 'pandas'